# Tabular to Text — Human-readable Sentences

Convert CSV rows into human-readable sentences and save a new CSV containing a `text` column and optional target column (`Fertilizer`).

In [ ]:
from pathlib import Path
import pandas as pd

def _pretty_col(col):
    # make column names human-friendly: replace underscores and trim
    return str(col).replace('_', ' ').strip().lower().capitalize()

def serialize_row_sentence(row, exclude=set()):
    parts = []
    for col, val in row.items():
        if col in exclude:
            continue
        if pd.isna(val):
            continue
        s = str(val).replace('\n', ' ').strip()
        if s == '':
            continue
        human_col = _pretty_col(col)
        parts.append(f"{human_col} is {s}")
    if not parts:
        return ''
    sentence = ', '.join(parts).strip()
    # Capitalize first letter and ensure period at end
    if not sentence.endswith('.'):
        sentence = sentence.rstrip('.') + '.'
    return sentence

def convert_table_to_text(input_path, output_path, target=None, sentence_mode=True):
    input_path = Path(input_path)
    output_path = Path(output_path)
    df = pd.read_csv(input_path)

    if target and target not in df.columns:
        print(f"Warning: target column '{target}' not found in input CSV.")
        target = None

    exclude = {target} if target else set()

    if sentence_mode:
        df['text'] = df.apply(lambda r: serialize_row_sentence(r, exclude=exclude), axis=1)
    else:
        # fallback to terse KV style if requested
        df['text'] = df.apply(lambda r: ', '.join([f"{c}: {str(v).replace('\n',' ').strip()}" for c,v in r.items() if c not in exclude and not pd.isna(v)]), axis=1)

    if target:
        out_df = df[['text', target]]
    else:
        out_df = df[['text']]

    out_df.to_csv(output_path, index=False)
    print(f"Serialized dataset saved to: {output_path}")

In [ ]:
# Example: change paths as needed
input_path = r"C:/Users/AN TECH BD/OneDrive/Desktop/Macine Learning Project/Crop_Prediction.csv"
output_path = r"C:/Users/AN TECH BD/OneDrive/Desktop/Macine Learning Project/Crop_Prediction_text_Human_Readable.csv"
target = 'Fertilizer'

# Convert using human-readable sentence mode (default)
convert_table_to_text(input_path, output_path, target, sentence_mode=True)

# Preview the first few lines
print(pd.read_csv(output_path).head().to_string())

In [9]:
from pathlib import Path
import pandas as pd

def serialize_agri_row(row, exclude=set()):
    parts = []

    if not pd.isna(row.get("Temperature")):
        parts.append(f"the temperature is {row['Temperature']}°C")

    if not pd.isna(row.get("Moisture")):
        parts.append(f"soil moisture is {row['Moisture']}%")

    if not pd.isna(row.get("Rainfall")):
        parts.append(f"rainfall is {row['Rainfall']} mm")

    if not pd.isna(row.get("PH")):
        parts.append(f"soil pH is {row['PH']}")

    if not pd.isna(row.get("Nitrogen")):
        parts.append(f"nitrogen level is {row['Nitrogen']}")

    if not pd.isna(row.get("Phosphorous")):
        parts.append(f"phosphorous level is {row['Phosphorous']}")

    if not pd.isna(row.get("Potassium")):
        parts.append(f"potassium level is {row['Potassium']}")

    if not pd.isna(row.get("Carbon")):
        parts.append(f"carbon content is {row['Carbon']}")

    if not pd.isna(row.get("Soil")):
        parts.append(f"soil type is {row['Soil']}")

    if not pd.isna(row.get("Crop")):
        parts.append(f"the crop is {row['Crop']}")

    if not pd.isna(row.get("Remark")):
        parts.append(f"remark: {row['Remark']}")

    # combine nicely
    if len(parts) > 1:
        text = ", ".join(parts[:-1]) + ", and " + parts[-1]
    elif parts:
        text = parts[0]
    else:
        text = ""

    return "In this condition, " + text + "."


def convert_table_to_human_text(input_path, output_path, target=None):
    df = pd.read_csv(input_path)

    exclude = {target} if target else set()

    df['text'] = df.apply(lambda r: serialize_agri_row(r, exclude=exclude), axis=1)

    if target:
        out_df = df[['text', target]]
    else:
        out_df = df[['text']]

    out_df.to_csv(output_path, index=False)
    print(f"Human-readable dataset saved to: {output_path}")

In [10]:
import pandas as pd

# Example: change paths as needed
input_path = r"C:/Users/AN TECH BD/OneDrive/Desktop/Macine Learning Project/Crop_Prediction.csv"
output_path = r"C:/Users/AN TECH BD/OneDrive/Desktop/Macine Learning Project/Crop_Prediction_text_Human_Readable.csv"
target = 'Fertilizer'

# Convert using human-readable sentence mode
convert_table_to_human_text(input_path, output_path, target)

# Preview the first few lines
print(pd.read_csv(output_path).head().to_string())

Human-readable dataset saved to: C:/Users/AN TECH BD/OneDrive/Desktop/Macine Learning Project/Crop_Prediction_text_Human_Readable.csv
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      text                  Fertilizer
0                                     In this condition, the temperature is 50.17984508°C, soil moisture is 0.725892978%, rainfall is 205.600816 mm, soil pH is 6.227357805, nitrogen level is 66.70187235, phosphorous level is 76.96356027, potassium level is 96.42906521, carbon content is 0.496299708, soil type is Loamy Soil, the crop is rice, and remark: Enhance

**Notes:**
- The serializer produces sentences like: 'Soil type is Sandy, Temperature is 30.'
- Set `sentence_mode=False` to get terse `key: value` serialization instead.
- Install pandas with `pip install pandas` if needed.

In [13]:
import pandas as pd

# -----------------------------
# Knowledge Functions
# -----------------------------

def interpret_ph(ph):
    if ph < 5.5:
        return "The soil is highly acidic which may limit nutrient availability for many crops."
    elif ph < 6.5:
        return "The soil is slightly acidic which is generally suitable for most crops."
    elif ph <= 7.5:
        return "The soil is neutral and considered ideal for a wide range of crops."
    else:
        return "The soil is alkaline which may reduce the availability of certain micronutrients."

def interpret_moisture(m):
    if m < 30:
        return "The soil is very dry and irrigation may be required."
    elif m < 60:
        return "The soil has moderate moisture levels suitable for crop growth."
    else:
        return "The soil has high moisture content which may increase the risk of waterlogging."

def interpret_rainfall(r):
    if r < 50:
        return "The region receives low rainfall and may require irrigation support."
    elif r < 150:
        return "The region receives moderate rainfall which supports most crops."
    else:
        return "The region receives high rainfall and proper drainage may be important."

def interpret_npk(n, p, k):
    notes = []

    if n < 40:
        notes.append("Nitrogen levels are low which may lead to poor vegetative growth.")
    elif n > 80:
        notes.append("Nitrogen levels are high which may cause excessive leaf growth.")

    if p < 30:
        notes.append("Phosphorous levels are low which may affect root and flower development.")
    elif p > 70:
        notes.append("Phosphorous levels are high which may create nutrient imbalance.")

    if k < 30:
        notes.append("Potassium levels are low which may reduce plant resistance to disease.")
    elif k > 70:
        notes.append("Potassium levels are high which may interfere with other nutrient uptake.")

    if not notes:
        return "The NPK levels are balanced and suitable for healthy crop growth."

    return " ".join(notes)

# -----------------------------
# Semantic Serializer (Paragraph Style)
# -----------------------------

def serialize_row_semantic(row):
    parts = []

    if not pd.isna(row.get("Temperature")):
        parts.append(f"The temperature is {row['Temperature']} degrees Celsius.")

    if not pd.isna(row.get("Moisture")):
        parts.append(f"The soil moisture is {row['Moisture']} percent. {interpret_moisture(row['Moisture'])}")

    if not pd.isna(row.get("Rainfall")):
        parts.append(f"The rainfall is {row['Rainfall']} millimeters. {interpret_rainfall(row['Rainfall'])}")

    if not pd.isna(row.get("PH")):
        parts.append(f"The soil pH is {row['PH']}. {interpret_ph(row['PH'])}")

    if not pd.isna(row.get("Nitrogen")) and not pd.isna(row.get("Phosphorous")) and not pd.isna(row.get("Potassium")):
        parts.append(
            f"The soil contains nitrogen level {row['Nitrogen']}, phosphorous level {row['Phosphorous']}, and potassium level {row['Potassium']}. {interpret_npk(row['Nitrogen'], row['Phosphorous'], row['Potassium'])}"
        )

    if not pd.isna(row.get("Carbon")):
        parts.append(f"The carbon content of the soil is {row['Carbon']} which reflects organic matter presence.")

    if not pd.isna(row.get("Soil")):
        parts.append(f"The soil type is {row['Soil']}.")

    if not pd.isna(row.get("Crop")):
        parts.append(f"The crop being considered is {row['Crop']}.")

    if not pd.isna(row.get("Fertilizer")):
        parts.append(f"The recommended fertilizer for this condition is {row['Fertilizer']}.")

    if not pd.isna(row.get("Remark")):
        parts.append(f"Additional observation: {row['Remark']}.")

    return " ".join(parts)


# -----------------------------
# Conversion Function
# -----------------------------

def convert_table_to_semantic_text(input_path, output_path, target=None):
    df = pd.read_csv(input_path)

    df["text"] = df.apply(serialize_row_semantic, axis=1)

    if target:
        out_df = df[["text", target]]
    else:
        out_df = df[["text"]]

    out_df.to_csv(output_path, index=False)
    print(f"Semantic enriched dataset saved to: {output_path}")



In [15]:
# -----------------------------
# Example usage
# -----------------------------
if __name__ == "__main__":
    input_path = r"C:/Users/AN TECH BD/OneDrive/Desktop/Macine Learning Project/Crop_Prediction.csv"
    output_path = r"C:/Users/AN TECH BD/OneDrive/Desktop/Macine Learning Project/Crop_Prediction_semantic_K.csv"
    target = "Fertilizer"

    convert_table_to_semantic_text(input_path, output_path, target)

    print(pd.read_csv(output_path).head().to_string())

Semantic enriched dataset saved to: C:/Users/AN TECH BD/OneDrive/Desktop/Macine Learning Project/Crop_Prediction_semantic_K.csv
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        